# Pressão disjoining
Da teoria DLVO, o cálculo da pressão disjoining ${\Pi _t}$, com base na distância de separação $(h)$ entre as superfícies, é dada pela decomposição em três componentes: pressão de Van der Waal $\left( {{\Pi _{vdw}}} \right)$, pressão da dupla camada elétrica $\left( {{\Pi _{edl}}} \right)$ e pressão estrutural $\left( {{\Pi _{str}}} \right)$,

$${\Pi _t}\left( h \right) = {\Pi _{vdw}}\left( h \right) + {\Pi _{edl}}\left( h \right) + {\Pi _{str}}\left( h \right)$$

onde:
$\mathrm{\Pi}_{vdw}\left(h\right)\rightarrow$ pressão de Van der Waals
$\mathrm{\Pi}_{edl}\left(h\right)\rightarrow$ pressão de dupla camada elétrica
$\mathrm{\Pi}_{str}\left(h\right)\rightarrow$  pressão da força estrutural

Ao substituir os diferentes tipos de pressões, denominadas pressão de Van der Waal $(\mathrm{\Pi}_{vdw}\left(h\right))$, pressão estrutural $(\mathrm{\Pi}_{str}\left(h\right))$ e pressão da dupla camada elétrica $(\mathrm{\Pi}_{edl}\left(h\right))$, obtém-se:

$$\mathrm{\Pi}_t\left(h\right)=A_sexp\left(-\frac{h}{h_s}\right)-\frac{A_h}{6\pi h^3}+\frac{\varepsilon_0\varepsilon\kappa^2\left[2V_1V_2\cosh{\left(\kappa h\right)}-\left(V_1^2+V_2^2\right)\right]}{2\left[\mathrm{sinh}\left(\kappa h\right)\right]^2}$$

sendo

$$\kappa=\left(\frac{2e^2N_AI}{\varepsilon_r\varepsilon_0k_BT}\right)^{1/2}$$

onde $V_1$ e $V_2$ são os potenciais das superfícies do óleo e da rocha dado em Volts, $e$  é a carga elementar $(1,6\times{10}^{-19}\ \mathrm{\mathrm{C}})$, $\varepsilon_0$ é a permissividade do vácuo $\left(8,8\times{10}^{-12}\mathrm{F/m}\ \right)$, $\varepsilon_r$ é a constante dielétrica relativa da salmoura $\left(\varepsilon_r=80\right)$, $k_B$ é a constante de Boltzmann $(1,38\times{10}^{-23}\ J/K)$, $N_A$ é o número de Avogadro $\left(6,0\times{10}^{23}{\mathrm{mol}}^{\mathrm{-1}}\right)$ e $I$ é a força iônica ($\mathrm{M}$ ou $\mathrm{mol/l}$).

In [1]:
!pip install engineering-notation

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/


In [3]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib notebook
from engineering_notation import EngNumber
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.signal import argrelextrema

In [4]:
def constants():
    global Tk, Pr, F_cal, kappa, A_ST, h_STR, PI, K_b, H_p, F_UV, elec, N_A, epson_0, F_dol, F_qua
    A_ST = 1.5e10  # Structural coefficient (Pa)
    h_STR = 5.0e-11  # Length decay characteristic (m)
    PI = np.pi  # PI number
    K_b = 1.380649e-23  # Boltzmann constant (m 2 Kg s-2 K-1)
    H_p = 6.62607015e-34  # Planck constant (m2 kg s-1)
    F_UV = 3.0e15  # Main electric adsorption frequency (s-1)
    elec = 1.602217663e-19  # Electronic charge (C)
    N_A = 6.02214076e23  # Avogadro number (mol-1)
    epson_0 = 8.854e-12  # Vacuum permittivity (F m-1)

In [5]:
def entrada():
    global Tk, Pr, F_cal, F_dol, F_qua, density_cal, density_dol, density_qua, density_oil, I_ion, IFT, Psi_1, Psi_2, xhmin, xhmax, Prange, erro, N

    # Magnitudes de Estado
    Tk = 338.0
    Pr = 0.01

    # Saídas do Prheeeqc
    I_ion = 10e-3
    Psi_1 = -50e-3
    Psi_2 = -50e-3

    # Entradas
    density_cal = 2.6
    density_dol = 1.8
    density_qua = 2.65 # (g/cm³)
    density_oil = 0.846 # (g/cm³)
    IFT = 25e-3


    # Rock cmposition
    F_cal = 0.55
    F_dol = 0.38
    F_qua = 0.07

    # Maximum thickness
    xhmax = 2.0e-9
    xhmin = 0.01e-9
    Prange = int(10e3)
    erro = 1e-6
    N = 1000

$${\Pi _{edl}}\left( h \right) = \frac{{{\varepsilon _0}\varepsilon {\kappa ^2}\left[ {2{V_1}{V_2}\cosh \left( {\kappa h} \right) - \left( {V_1^2 + V_2^2} \right)} \right]}}{{2{{\left[ {{\rm{sinh}}\left( {\kappa h} \right)} \right]}^2}}}$$

In [6]:
def piedl(xh, Psi_1, Psi_2, epson_3, kappa):
    temp_1 = epson_0 * epson_3 * (kappa * kappa)
    temp_2 = 2.0 * Psi_1 * Psi_2 * np.cosh(kappa * xh) - (Psi_1**2 + Psi_2**2)
    temp_3 = 2.0 * np.sinh(kappa * xh) * np.sinh(kappa * xh)
    temp_4 = temp_1 * temp_2 / temp_3
    return temp_4

In [ ]:
def f(x0):
    global kappa, epson_0, epson_3, Psi_1, Psi_2, PI
    t1 = epson_0 * epson_3 * kappa ** 2
    t2 = 2 * Psi_1 * Psi_2
    t3 = Psi_1**2 + Psi_2**2
    t4 = A_HA / (6 * PI)

    return t1 * (t2 * np.cosh(kappa * x0) - t3) / (2 * np.arcsinh(kappa * x0) * np.arcsinh(kappa * x0)) - t4 / (
                x0 * x0 * x0) + A_ST * np.exp(-x0 / h_STR)

In [ ]:
def g(x0):
    global t5, t6, t7, Psi_1, Psi_2, kappa, h_STR
    t5 = A_HA / (2 * PI)
    t6 = A_ST / h_STR
    t7 = epson_0 * epson_3 * kappa ** 2
    return t5 / (x0 * x0 * x0 * x0) - t6 * np.exp(-x0 / h_STR) - (
                t7 / (np.arcsinh(kappa * x0) * np.arcsinh(kappa * x0) * np.arcsinh(kappa * x0))) * (
                Psi_2 * np.cosh(kappa * x0) - Psi_1) * (Psi_1 * np.cosh(kappa * x0) - Psi_2)

In [ ]:
def newton_raphson(x0, N):
    cont = 0
    while True:
        f0 = f(x0)
        g0 = g(x0)

        if (g0 == 0):
            print('Mathematical Error.')
            continua = False
            break

        x1 = x0 - f0 / g0
        x0 = x1
        f1 = f(x1)

        cont += 1
        if cont > N:
            continua = False
            break

        if abs(f1) <= erro:
            continua = True
            break
    return [x1, continua]

In [ ]:
def calcular_permissividades_refracao_hamaker_kappa():
    global epson_1, epson_2, epson_3, kappa, n_1, n_2, n_3, A_HA
    epson_3 = 6 * I_ion * (0.08 * I_ion - 1) - 0.165 * (Tk - 273) + 0.019 * Pr + 82.5
    epson_2 = F_cal * (6 * density_cal - 8.1) + F_dol * (1.9 * density_dol) + F_qua * (1.27 * density_qua + 1.25)
    epson_1 = 1.34 + 0.96 * density_oil + 0.015 * np.log(10 * Pr) - 0.00035 * (Tk - 273)
    n_1 = 0.0006 * (Tk - 273) + 0.0004 * Pr + 1.45
    n_2 = 1.724 * F_cal + 1.734 * F_dol + 1.793 * F_qua
    n_temp = 1.34 + 0.13 * Pr - 1.56 * (10 ** -4) * (Tk - 273)
    n_3 = n_temp + I_ion * (2 * (10 ** -1) - 8 * (10 ** -4) * (Tk - 273))
    # -------------------- Calculation constant Hamaker f=0 ----------------------------
    A_HF0 = K_b * Tk * ((epson_1 - epson_2) / (epson_1 + epson_2)) * ((epson_2 - epson_3) / (epson_2 + epson_3))
    # -------------------- Calculation constant Hamaker f>0 ----------------------------
    temp_1 = (3 * H_p * F_UV) / (8 * np.sqrt(2))
    temp_2 = (n_1 * n_1 - n_3 * n_3) * (n_2 * n_2 - n_3 * n_3)
    temp_3 = np.sqrt((n_1 * n_1 + n_3 * n_3) * (n_2 * n_2 + n_3 * n_3))
    temp_4 = (np.sqrt(n_1 * n_1 + n_3 * n_3) + np.sqrt(n_2 * n_2 + n_3 * n_3))
    A_HF1 = temp_1 * (temp_2) / ((temp_3) * (temp_4))
    # -------------------- Calculation constant Hamaker A_HA ---------------------------
    A_HA = A_HF0 + A_HF1
    # -------------------- Calculation Debye lenght ------------------------------------
    kappa = np.sqrt(((2.0e03 * elec * elec * N_A) / (epson_0 * epson_3 * K_b)) * (I_ion / Tk))
    # -------------------------- calculation thickness equilibrium --------------------


# Cálculo da espessura de equilíbrio $h_0$

No contexto do equilíbrio termodinâmico entre a formação, salmoura e óleo, usa-se a equação de Young-Laplace aumentada, expressa por:

$$P_c=\mathrm{\Pi}_t+2C_m\sigma$$

onde $P_c$ é a pressão capilar, $\mathrm{\Pi}_t$ é a pressão disjoining, $C_m$ é a curvatura média e $\sigma$ é a tensão interfacial entre as fases de óleo e água. Para calcular a espessura de equilíbrio $h_0$, basta fazer $C_m=0$:

$$P_c=\mathrm{\Pi}_t$$

Como região de contato trifásica, as forças de superfície são dominantes, é possível desprezar as forças capilares $\left(\mathrm{\Pi}_t=0\right)$:

$$A_sexp\left(-\frac{h_0}{h_s}\right)-\frac{A_h}{6\pi h_0^3}+\frac{\varepsilon_0\varepsilon\kappa^2\left[2V_1V_2\cosh{\left(\kappa h_0\right)}-\left(V_1^2+V_2^2\right)\right]}{2\left[\mathrm{sinh}\left(\kappa h_0\right)\right]^2}=0$$

A solução da equação fornece a espessura de equilíbrio $h_0$.

In [ ]:
def expessura_de_equilibrio():
    global root
    x0 = xhmin
    x1, continua = newton_raphson(x0, N)
    global xh, PI_ST, PI_VdW, PI_EDL, PI_TOT
    if continua:
        root = x1
        print('Equilibrium thickness |  h_eq =  ', str(EngNumber(root)), 'm')
    else:
        theta = 'theta error'
        print('Not Convergent.')




In [ ]:
def calculo_das_pressoes():
    global xh, PI_ST, PI_VdW, PI_VdW, PI_EDL, PI_TOT
    # Estrutural
    xh = np.geomspace(xhmin, xhmax, num=Prange)
    # xh = np.linspace(np.log10(xhmin), np.log10(xhmax), Prange)
    PI_ST = A_ST * np.exp(-xh / h_STR)
    # van der walls
    PI_VdW = -A_HA / (6 * PI * (xh * xh * xh))
    # double layer
    PI_EDL = piedl(xh, Psi_1, Psi_2, epson_3, kappa)
    # total
    PI_TOT = PI_VdW + PI_ST + PI_EDL



# Cálculo do angulo de contato

No que segue, integrando a equação de Young-Laplace aumentada, é possível obter a seguinte relação entre o ângulo de contato estático:

$$\theta_e=\cos^{-1}{\left[1+X_{edl}+X_{vdw}+X_s\right]}$$

onde:
$X_{edl}=\frac{\varepsilon_0\varepsilon\kappa^2}{2\sigma}\left\{-V_1V_2\left[\coth{\left(\kappa h_0\right)}\right]^2-\left(V_1^2+V_2^2\right)\csch{\left(\kappa h_0\right)}\left\{\left[\coth{\left(\kappa h_0\right)}\right]^2-\frac{\left[\csch{\left(\kappa h_0\right)}\right]^2}{3}\right\}\right\}$
$X_{vdw}=-\frac{A_h}{12\pi\sigma h_0^2}$
$X_s=\frac{A_sh_s}{\sigma}exp\left(-\frac{h_0}{h_s}\right)$

In [ ]:
def calculo_do_angulo():
    global theta_graus
    temp5 = epson_0 * epson_3 * kappa * Psi_1 * Psi_2 / IFT
    temp6 = epson_0 * epson_3 * kappa * ((Psi_1 * Psi_1) + (Psi_2 * Psi_2)) / (2 * IFT)
    XEDL = temp5 / np.sinh(kappa * root) - temp6 * np.cosh(kappa * root) / np.sinh(kappa * root)
    XVDW = -A_HA / (12. * PI * IFT * (root * root))
    fit = 932
    XST = fit * A_ST * h_STR * IFT * np.exp(-root / h_STR)
    ARG = 1.0 + XEDL + XVDW + XST
    theta_graus = (180 * np.arccos(ARG)) / PI



In [ ]:
def find_extremos(vector):
    local_max = None
    local_min = None

    for i in range(1, len(vector) - 1):  # Vamos ignorar os pontos finais
        # Encontre o máximo local
        if vector[i - 1] < vector[i] > vector[i + 1]:
            if local_max is None or vector[i] > vector[local_max]:
                local_max = i

        # Encontre o mínimo local
        if vector[i - 1] > vector[i] < vector[i + 1]:
            if local_min is None or vector[i] < vector[local_min]:
                local_min = i

    return (vector[local_max], vector[local_min]) if local_max is not None and local_min is not None else None

In [ ]:
def graficos_pessoes():
    fig = make_subplots(rows=2, cols=2)

    fig.add_trace(go.Scatter(x=xh, y=PI_ST, mode='lines', name='Structural'), row=1, col=1)
    fig.add_trace(go.Scatter(x=xh, y=PI_VdW, mode='lines', name='Van der Walls'), row=1, col=2)
    fig.add_trace(go.Scatter(x=xh, y=PI_EDL, mode='lines', name='Electric Double Layeer'), row=2, col=1)
    fig.add_trace(go.Scatter(x=xh, y=PI_TOT, mode='lines', name='Total'), row=2, col=2)

    fig.update_yaxes(title_text="Pressão [N/m²]", title_font=dict(size=10), row=1, col=1)
    # fig.update_yaxes(title_text="Pressão [N/m²]", title_font=dict(size=10), row=1, col=2)
    fig.update_yaxes(title_text="Pressão [N/m²]", title_font=dict(size=10), row=2, col=1)
    # fig.update_yaxes(title_text="Pressão [N/m²]", title_font=dict(size=10), row=2, col=2)

    fig.update_xaxes(title_text="Posição [m]", title_font=dict(size=10), row=2, col=1)
    fig.update_xaxes(title_text="Posição [m]", title_font=dict(size=10), row=2, col=2)

    lim = find_extremos(PI_TOT)
    if lim is not None:
        fig.update_yaxes(range=[lim[0], lim[1]], row=2, col=2)

    return fig


In [ ]:
constants()
entrada()
calcular_permissividades_refracao_hamaker_kappa()
expessura_de_equilibrio()
calculo_das_pressoes()
calculo_do_angulo()
graficos_pessoes().show()

Equilibrium thickness |  h_eq =   58.29p m


In [ ]:
theta_graus

5.068327451766758